In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/suchintikasarkar/sentiment-analysis-for-mental-health/Combined Data.csv


### Preprocessing

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv("/kaggle/input/datasets/suchintikasarkar/sentiment-analysis-for-mental-health/Combined Data.csv")
df = df.dropna(subset=["statement", "status"]).drop_duplicates()

le = LabelEncoder()
df["label"] = le.fit_transform(df["status"])
num_labels = df["label"].nunique()

# Divisione del Dataset in train e validation
train_df, val_df = train_test_split(
    df, test_size=0.15, stratify=df["label"], random_state=42
)

In [3]:
# Stampa metriche dei Datafarme creati

print(f"Train DataFrame size: {train_df.shape}")
print(f"Train DataFrame samples:\n {train_df.head()}")
print(f"\nVal DataFrame size: {val_df.shape}")
print(f"Val DataFrame samples:\n {val_df.head()}")

Train DataFrame size: (44778, 4)
Train DataFrame samples:
        Unnamed: 0                                          statement  \
42636       42636                                     congested nose   
15210       15210  Please anyone who has been in the same situati...   
43578       43578  yopatrizzle not sure to tell u the truth it s ...   
7224         7224  I went to my psych about a week or so ago for ...   
14506       14506  I am not sure why, but i feel empty and discon...   

           status  label  
42636      Normal      3  
15210  Depression      2  
43578      Normal      3  
7224   Depression      2  
14506  Depression      2  

Val DataFrame size: (7903, 4)
Val DataFrame samples:
        Unnamed: 0                                          statement  \
7486         7486  I do not know how else to say this. My air con...   
51809       51809  crippling guilt from my inability to answer te...   
23780       23780  Therapy is moving slowly because I cannot fuck...   
105

### Creazione Dataset da CSV

In [4]:
from transformers import AutoTokenizer
from datasets import Dataset

# Recupero del tokenizer del modello da usare
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Funzione per tokenizzare un batch testuale 
def tokenize(batch):
    return tokenizer(batch["statement"], truncation=True, padding="max_length", max_length=256)

# Applicazione del tokenizer ai dataset train e validation
train_ds = Dataset.from_pandas(train_df[["statement", "label"]]).map(tokenize, batched=True)
val_ds = Dataset.from_pandas(val_df[["statement", "label"]]).map(tokenize, batched=True)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/44778 [00:00<?, ? examples/s]

Map:   0%|          | 0/7903 [00:00<?, ? examples/s]

In [5]:
# Stampa delle dimensioni dei dataset dopo l'applicazione del tokenizer

print(f"Train Dataset size: {train_ds.shape}")
print(f"Train Dataset sample:\n {train_ds[0]}")
print(f"Val Dataset size: {val_ds.shape}")
print(f"Val Dataset sample:\n {val_ds[0]}")

Train Dataset size: (44778, 6)
Train Dataset sample:
 {'statement': 'congested nose', 'label': 3, '__index_level_0__': 42636, 'input_ids': [101, 26478, 17944, 4451, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [6]:
from transformers import AutoModelForSequenceClassification

# Caricamento pesi del modello pre-trainato e aggiunta di un layer Head non addestrato per la classificazione 
# (num_labels indica i nodi finali della classificazione)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=num_labels
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# Calcolo delle metriche di valutazione partendo dalle predizioni
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro")
    }

# Argomenti per il train
args = TrainingArguments(
    output_dir="./results", # Cartella dove salvare checkpoint e logs
    eval_strategy="epoch", # modello valutato sul validation set alla fine di ogni epoca
    save_strategy="epoch", # modello salvato dopo ogni epoca
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3, # numero di passaggi completi sull'intero training set
    weight_decay=0.01,
    load_best_model_at_end=True, # vengono caricati i pesi del checkpoint con prestazioni migliori
    metric_for_best_model="f1_macro", # metrica da usare per la valutazione del miglior checkpoint
    fp16=True,  # abilita il training in mixed precision (calcoli in float16 invece di float32 dove possibile)
)

# Creazione dell'oggetto Trainer
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

# Addestramento (Fine-tuning) del modello
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.049828,0.960513,0.805770,0.784113
2,0.785889,0.838820,0.829432,0.810181
